iNat-Aves experiments use an order–family–genus–species taxonomy, frozen ResNet-50 features, and a trainable shared projection and heads. Configure the `aves_features.npz` and checkpoint paths before running. See `docs/TRAINING.md`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
CKPT_DIR = "/content/drive/MyDrive/resolvfl_inat_aves"
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoints en:", CKPT_DIR)


In [ ]:
# Ruta al fichero que subiste. Ajusta si lo guardaste en otra carpeta.
FEATURES_PATH = "/content/drive/MyDrive/resolvfl_inat/aves_features.npz"
assert os.path.exists(FEATURES_PATH), f"No encuentro el fichero en {FEATURES_PATH} — revisa la ruta"
print("Fichero encontrado:", FEATURES_PATH)


In [ ]:
%%writefile resolvfl_colab.py
"""
resolvfl_colab.py — ResolvFL para CIFAR-100, listo para Colab.
Con soporte de GPU (device) y checkpointing en Google Drive para poder
parar y retomar entre sesiones sin perder progreso.
"""
from __future__ import annotations
import copy, json, os, time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Taxonomía (idéntica a la versión de validación, con certificado)
# ============================================================
class Taxonomy:
    ROOT = "ROOT"

    def __init__(self, parent: Dict[str, str]):
        self.parent = dict(parent)
        names = [self.ROOT] + list(parent.keys())
        self.name2id = {n: i for i, n in enumerate(names)}
        self.id2name = names
        self.children: Dict[int, List[int]] = {self.name2id[n]: [] for n in names}
        for c, p in parent.items():
            self.children[self.name2id[p]].append(self.name2id[c])
        self.leaves: List[int] = [i for i in range(len(names)) if not self.children[i]]
        self.internal: List[int] = [i for i in range(len(names)) if self.children[i]]
        self.leaf_index = {v: k for k, v in enumerate(self.leaves)}
        self.depth = {self.name2id[self.ROOT]: 0}
        self.path: Dict[int, List[int]] = {self.name2id[self.ROOT]: [self.name2id[self.ROOT]]}
        pending = [c for c in parent]
        while pending:
            nxt = []
            for c in pending:
                p = self.name2id[self.parent[c]]
                cid = self.name2id[c]
                if p in self.depth:
                    self.depth[cid] = self.depth[p] + 1
                    self.path[cid] = self.path[p] + [cid]
                else:
                    nxt.append(c)
            pending = nxt
        self.max_depth = max(self.depth.values())
        self.child_slot = {}
        for u in self.internal:
            for slot, c in enumerate(self.children[u]):
                self.child_slot[c] = slot
        self.leaves_under: Dict[int, List[int]] = {}
        for nid in range(len(names)):
            self.leaves_under[nid] = self._collect_leaves(nid)

    def _collect_leaves(self, nid: int) -> List[int]:
        if not self.children[nid]:
            return [nid]
        out = []
        for c in self.children[nid]:
            out += self._collect_leaves(c)
        return out

    def ancestor_at_depth(self, leaf_id: int, d: int) -> int:
        p = self.path[leaf_id]
        return p[min(d, len(p) - 1)]

    def path_edges(self, node_id: int) -> List[Tuple[int, int]]:
        p = self.path[node_id]
        return [(p[j], self.child_slot[p[j + 1]]) for j in range(len(p) - 1)]

    def certificate(self, histograms: List[Dict[int, float]],
                    weights: Optional[List[float]] = None) -> Dict:
        K = len(histograms)
        w = weights or [1.0 / K] * K
        agg = {}
        for wk, h in zip(w, histograms):
            for v, p in h.items():
                agg[v] = agg.get(v, 0.0) + wk * p
        m = {u: 0.0 for u in self.internal}
        order = sorted(range(len(self.id2name)), key=lambda n: -self.depth[n])
        below = {n: 0.0 for n in range(len(self.id2name))}
        for n in order:
            s = 0.0
            for c in self.children[n]:
                s += below[c] + agg.get(c, 0.0)
            below[n] = s
            if n in m:
                m[n] = s
        resolved = {u: m[u] > 0 for u in self.internal}
        maximal_unresolved = [u for u in self.internal
                              if not resolved[u]
                              and all(resolved.get(a, True)
                                      for a in self.path[u][:-1] if a in resolved)]
        blocked = set()
        for u in maximal_unresolved:
            blocked.update(self.leaves_under[u])
        partition = [[l] for l in self.leaves if l not in blocked]
        partition += [list(self.leaves_under[u]) for u in maximal_unresolved]
        return dict(mass=m, resolved=resolved,
                    maximal_unresolved=maximal_unresolved,
                    leaf_partition=partition)


COARSE_TO_FINE = {
    "aquatic_mammals": ["beaver", "dolphin", "otter", "seal", "whale"],
    "fish": ["aquarium_fish", "flatfish", "ray", "shark", "trout"],
    "flowers": ["orchid", "poppy", "rose", "sunflower", "tulip"],
    "food_containers": ["bottle", "bowl", "can", "cup", "plate"],
    "fruit_and_vegetables": ["apple", "mushroom", "orange", "pear", "sweet_pepper"],
    "household_electrical_devices": ["clock", "keyboard", "lamp", "telephone", "television"],
    "household_furniture": ["bed", "chair", "couch", "table", "wardrobe"],
    "insects": ["bee", "beetle", "butterfly", "caterpillar", "cockroach"],
    "large_carnivores": ["bear", "leopard", "lion", "tiger", "wolf"],
    "large_man-made_outdoor_things": ["bridge", "castle", "house", "road", "skyscraper"],
    "large_natural_outdoor_scenes": ["cloud", "forest", "mountain", "plain", "sea"],
    "large_omnivores_and_herbivores": ["camel", "cattle", "chimpanzee", "elephant", "kangaroo"],
    "medium_mammals": ["fox", "porcupine", "possum", "raccoon", "skunk"],
    "non-insect_invertebrates": ["crab", "lobster", "snail", "spider", "worm"],
    "people": ["baby", "boy", "girl", "man", "woman"],
    "reptiles": ["crocodile", "dinosaur", "lizard", "snake", "turtle"],
    "small_mammals": ["hamster", "mouse", "rabbit", "shrew", "squirrel"],
    "trees": ["maple_tree", "oak_tree", "palm_tree", "pine_tree", "willow_tree"],
    "vehicles_1": ["bicycle", "bus", "motorcycle", "pickup_truck", "train"],
    "vehicles_2": ["lawn_mower", "rocket", "streetcar", "tank", "tractor"],
}

def build_cifar100_taxonomy() -> Taxonomy:
    parent = {}
    for coarse, fines in COARSE_TO_FINE.items():
        parent[coarse] = Taxonomy.ROOT
        for f in fines:
            parent[f] = coarse
    return Taxonomy(parent)


# ============================================================
# Taxonomía real: Aves de iNat2021-mini (orden -> familia -> género -> especie)
# ============================================================
def build_aves_taxonomy(taxonomy_records: List[Dict[str, str]]):
    """A partir de la lista de dicts de taxonomía (uno por muestra, con
    claves order/family/genus/species), construye el árbol real de 4
    niveles. Las hojas usan 'Genero_especie' (no solo la especie), porque
    el epíteto específico puede repetirse en géneros distintos -- p.ej.
    'americana' aparece en varios géneros de aves.

    Devuelve (tax, leaf_name_per_sample): tax es la Taxonomy; leaf_name_per_sample
    es la lista de nombres de hoja, en el mismo orden que taxonomy_records,
    lista para mapear a tax.leaf_index.
    """
    parent = {}
    leaf_name_per_sample = []
    for rec in taxonomy_records:
        order, family, genus, species = rec["order"], rec["family"], rec["genus"], rec["species"]
        leaf_name = f"{genus}_{species}"
        parent.setdefault(order, Taxonomy.ROOT)
        parent.setdefault(family, order)
        parent.setdefault(genus, family)
        parent.setdefault(leaf_name, genus)
        leaf_name_per_sample.append(leaf_name)
    tax = Taxonomy(parent)
    return tax, leaf_name_per_sample


def load_aves_features(npz_path: str):
    """Carga el .npz producido por extract_features.py, construye la
    taxonomía real y devuelve (tax, X, y_leaf) listos para
    double_dirichlet_partition."""
    d = np.load(npz_path, allow_pickle=True)
    X = d["features"].astype(np.float32)
    taxonomy_records = list(d["taxonomy"])
    tax, leaf_names = build_aves_taxonomy(taxonomy_records)
    y_leaf = np.array([tax.leaf_index[tax.name2id[n]] for n in leaf_names])
    return tax, X, y_leaf


class FeatureBackbone(nn.Module):
    """Backbone ligero para features ya extraídas (frozen ResNet-50, 2048-dim).
    Esto SÍ se entrena/agrega en federado -- es la parte 'barata' que
    sustituye a ConvBackbone cuando las imágenes ya vienen convertidas
    a vectores."""
    def __init__(self, in_dim: int = 2048, feat_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(),
            nn.Linear(512, feat_dim), nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# Modelos
# ============================================================
class ConvBackbone(nn.Module):
    def __init__(self, feat_dim: int = 128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(128, feat_dim)

    def forward(self, x):
        z = self.conv(x).flatten(1)
        return self.fc(z)


class TreeHead(nn.Module):
    def __init__(self, tax: Taxonomy, feat_dim: int):
        super().__init__()
        self.tax = tax
        self.heads = nn.ModuleDict({
            str(u): nn.Linear(feat_dim, len(tax.children[u])) for u in tax.internal
        })

    def node_logits(self, z, u: int):
        return self.heads[str(u)](z)

    def leaf_log_probs(self, z) -> torch.Tensor:
        logp_node = {self.tax.name2id[Taxonomy.ROOT]: torch.zeros(z.shape[0], device=z.device)}
        order = sorted(self.tax.internal, key=lambda u: self.tax.depth[u])
        for u in order:
            lp = F.log_softmax(self.node_logits(z, u), dim=1)
            for slot, c in enumerate(self.tax.children[u]):
                logp_node[c] = logp_node[u] + lp[:, slot]
        return torch.stack([logp_node[l] for l in self.tax.leaves], dim=1)


class FlatHead(nn.Module):
    def __init__(self, tax: Taxonomy, feat_dim: int):
        super().__init__()
        self.tax = tax
        self.nodes = [n for n in range(len(tax.id2name)) if n != tax.name2id[Taxonomy.ROOT]]
        self.node_pos = {n: i for i, n in enumerate(self.nodes)}
        self.fc = nn.Linear(feat_dim, len(self.nodes))

    def forward(self, z):
        return self.fc(z)


# ============================================================
# Pérdida y agregación (device-aware)
# ============================================================
def marginalized_loss(tax: Taxonomy, backbone, head, x, y_nodes) -> torch.Tensor:
    z = backbone(x)
    device = x.device
    loss = x.new_zeros(())
    n = x.shape[0]
    groups: Dict[int, Tuple[List[int], List[int]]] = {}
    for i, yn in enumerate(y_nodes.tolist()):
        for u, slot in tax.path_edges(yn):
            groups.setdefault(u, ([], []))
            groups[u][0].append(i)
            groups[u][1].append(slot)
    for u, (idx, slots) in groups.items():
        logits = head.node_logits(z[idx], u)
        target = torch.tensor(slots, device=device)
        loss = loss + F.cross_entropy(logits, target, reduction="sum")
    return loss / n


def local_supervision_mass(tax: Taxonomy, y_nodes: np.ndarray) -> Dict[int, float]:
    n = len(y_nodes)
    m = {u: 0 for u in tax.internal}
    for yn in y_nodes:
        for u, _ in tax.path_edges(int(yn)):
            m[u] += 1
    return {u: c / n for u, c in m.items()}


def masked_aggregate(tax, client_states, n_samples, masses):
    total = float(sum(n_samples))
    w = [nk / total for nk in n_samples]
    bb = {key: sum(w[k] * client_states[k][0][key] for k in range(len(w)))
          for key in client_states[0][0]}
    hd = copy.deepcopy(client_states[0][1])
    for u in tax.internal:
        cohort = [k for k in range(len(w)) if masses[k].get(u, 0.0) > 0]
        if not cohort:
            continue
        omega = np.array([w[k] * masses[k][u] for k in cohort])
        omega = omega / omega.sum()
        for suffix in ("weight", "bias"):
            key = f"heads.{u}.{suffix}"
            hd[key] = sum(om * client_states[k][1][key] for om, k in zip(omega, cohort))
    return bb, hd


def plain_average(states, n_samples):
    total = float(sum(n_samples))
    w = [nk / total for nk in n_samples]
    return {key: sum(w[k] * states[k][key] for k in range(len(w))) for key in states[0]}


# ============================================================
# Partición double-Dirichlet
# ============================================================
def double_dirichlet_partition(tax, X, y_leaf, K, alpha_cls, alpha_dep, rng,
                               depth_override=None):
    n_leaves = len(tax.leaves)
    props = rng.dirichlet([alpha_cls] * K, size=n_leaves)
    client_idx = [[] for _ in range(K)]
    for li, leaf in enumerate(tax.leaves):
        idx = np.where(y_leaf == tax.leaf_index[leaf])[0]
        rng.shuffle(idx)
        cuts = (np.cumsum(props[li]) * len(idx)).astype(int)[:-1]
        for k, part in enumerate(np.split(idx, cuts)):
            client_idx[k].extend(part.tolist())
    clients = []
    for k in range(K):
        idx = np.array(sorted(client_idx[k]))
        nu = (depth_override or {}).get(k)
        if nu is None:
            nu = rng.dirichlet([alpha_dep] * tax.max_depth)
        depths = rng.choice(np.arange(1, tax.max_depth + 1), size=len(idx), p=nu)
        y_nodes = np.array([
            tax.ancestor_at_depth(tax.leaves[y_leaf[i]], d)
            for i, d in zip(idx, depths)
        ])
        clients.append(dict(X=X[idx], y_node=y_nodes, y_leaf=y_leaf[idx], nu=nu))
    return clients


# ============================================================
# Checkpointing en Drive (para poder parar y retomar)
# ============================================================
def save_checkpoint(path, round_num, backbone, head, extra=None):
    torch.save({
        "round": round_num,
        "backbone": backbone.state_dict(),
        "head": head.state_dict(),
        "extra": extra or {},
    }, path)


def load_checkpoint(path, backbone, head, map_location="cpu"):
    if not os.path.exists(path):
        return 0, {}
    ckpt = torch.load(path, map_location=map_location)
    backbone.load_state_dict(ckpt["backbone"])
    head.load_state_dict(ckpt["head"])
    return ckpt["round"], ckpt.get("extra", {})


# ============================================================
# Entrenamiento federado, con GPU y checkpoint/resume
# ============================================================
@dataclass
class FedConfig:
    rounds: int = 100
    local_epochs: int = 1
    batch_size: int = 128
    lr: float = 0.05
    feat_dim: int = 128
    seed: int = 0
    save_every: int = 5


def local_train_tree(tax, backbone, head, data, cfg, device):
    opt = torch.optim.SGD(list(backbone.parameters()) + list(head.parameters()),
                          lr=cfg.lr, momentum=0.9)
    X = torch.tensor(data["X"], device=device)
    y = torch.tensor(data["y_node"], device=device)
    n = len(y)
    last_loss = 0.0
    for _ in range(cfg.local_epochs):
        perm = torch.randperm(n, device=device)
        for s in range(0, n, cfg.batch_size):
            b = perm[s:s + cfg.batch_size]
            opt.zero_grad()
            loss = marginalized_loss(tax, backbone, head, X[b], y[b])
            loss.backward()
            opt.step()
            last_loss = loss.item()
    return last_loss


def run_resolvfl(tax, clients, cfg: FedConfig, device, ckpt_path=None,
                 progress_every=5, backbone_factory=None):
    """backbone_factory: función (feat_dim) -> nn.Module. Por defecto
    ConvBackbone (imágenes en bruto, como CIFAR-100). Para datos ya
    convertidos a vectores (p.ej. iNat-Aves con features de ResNet-50
    congelado), pasar backbone_factory=lambda d: FeatureBackbone(in_dim, d)."""
    if backbone_factory is None:
        backbone_factory = ConvBackbone
    backbone = backbone_factory(cfg.feat_dim).to(device)
    head = TreeHead(tax, cfg.feat_dim).to(device)
    masses = [local_supervision_mass(tax, c["y_node"]) for c in clients]
    n_samples = [len(c["y_node"]) for c in clients]

    start_round = 0
    if ckpt_path is not None:
        start_round, _ = load_checkpoint(ckpt_path, backbone, head, map_location=device)
        if start_round > 0:
            print(f"    [checkpoint] retomando ResolvFL desde la ronda {start_round}")

    t0 = time.time()
    for rnd in range(start_round, cfg.rounds):
        states, losses = [], []
        for c in clients:
            bb, hd = copy.deepcopy(backbone), copy.deepcopy(head)
            l = local_train_tree(tax, bb, hd, c, cfg, device)
            losses.append(l)
            states.append(({k: v.detach().clone() for k, v in bb.state_dict().items()},
                           {k: v.detach().clone() for k, v in hd.state_dict().items()}))
        bb_sd, hd_sd = masked_aggregate(tax, states, n_samples, masses)
        backbone.load_state_dict(bb_sd)
        head.load_state_dict(hd_sd)

        if ckpt_path is not None and (rnd + 1) % cfg.save_every == 0:
            save_checkpoint(ckpt_path, rnd + 1, backbone, head)

        if rnd % progress_every == 0 or rnd == cfg.rounds - 1:
            elapsed = time.time() - t0
            print(f"    [ResolvFL] ronda {rnd+1:3d}/{cfg.rounds} | "
                  f"loss medio={np.mean(losses):.4f} | "
                  f"{elapsed:.0f}s transcurridos")

    if ckpt_path is not None:
        save_checkpoint(ckpt_path, cfg.rounds, backbone, head)
    return backbone, head


def run_flat_fedavg(tax, clients, cfg: FedConfig, device, ckpt_path=None,
                    progress_every=5, backbone_factory=None):
    if backbone_factory is None:
        backbone_factory = ConvBackbone
    backbone = backbone_factory(cfg.feat_dim).to(device)
    head = FlatHead(tax, cfg.feat_dim).to(device)
    n_samples = [len(c["y_node"]) for c in clients]

    start_round = 0
    if ckpt_path is not None:
        start_round, _ = load_checkpoint(ckpt_path, backbone, head, map_location=device)
        if start_round > 0:
            print(f"    [checkpoint] retomando FedAvg-flat desde la ronda {start_round}")

    t0 = time.time()
    for rnd in range(start_round, cfg.rounds):
        bb_states, hd_states, losses = [], [], []
        for c in clients:
            bb, hd = copy.deepcopy(backbone), copy.deepcopy(head)
            opt = torch.optim.SGD(list(bb.parameters()) + list(hd.parameters()),
                                  lr=cfg.lr, momentum=0.9)
            X = torch.tensor(c["X"], device=device)
            y = torch.tensor([hd.node_pos[int(v)] for v in c["y_node"]], device=device)
            n = len(y)
            last_loss = 0.0
            for _ in range(cfg.local_epochs):
                perm = torch.randperm(n, device=device)
                for s in range(0, n, cfg.batch_size):
                    b = perm[s:s + cfg.batch_size]
                    opt.zero_grad()
                    loss = F.cross_entropy(hd(bb(X[b])), y[b])
                    loss.backward()
                    opt.step()
                    last_loss = loss.item()
            losses.append(last_loss)
            bb_states.append({k: v.detach().clone() for k, v in bb.state_dict().items()})
            hd_states.append({k: v.detach().clone() for k, v in hd.state_dict().items()})
        backbone.load_state_dict(plain_average(bb_states, n_samples))
        head.load_state_dict(plain_average(hd_states, n_samples))

        if ckpt_path is not None and (rnd + 1) % cfg.save_every == 0:
            save_checkpoint(ckpt_path, rnd + 1, backbone, head)

        if rnd % progress_every == 0 or rnd == cfg.rounds - 1:
            elapsed = time.time() - t0
            print(f"    [FedAvg-flat] ronda {rnd+1:3d}/{cfg.rounds} | "
                  f"loss medio={np.mean(losses):.4f} | "
                  f"{elapsed:.0f}s transcurridos")

    if ckpt_path is not None:
        save_checkpoint(ckpt_path, cfg.rounds, backbone, head)
    return backbone, head


# ============================================================
# Evaluación
# ============================================================
@torch.no_grad()
def eval_tree(tax, backbone, head, X, y_leaf, device):
    z = backbone(torch.tensor(X, device=device))
    lp = head.leaf_log_probs(z).cpu().numpy()
    pred = lp.argmax(1)
    acc = float((pred == y_leaf).mean())
    return dict(leaf_acc=acc, pred=pred)


@torch.no_grad()
def eval_flat(tax, backbone, head, X, y_leaf, device):
    z = backbone(torch.tensor(X, device=device))
    logits = head(z)
    p = F.softmax(logits, dim=1).cpu().numpy()
    node_pred = p.argmax(1)
    leaf_positions = [head.node_pos[l] for l in tax.leaves]
    abstain = float(np.mean([head.nodes[i] not in tax.leaves for i in node_pred]))
    pred = p[:, leaf_positions].argmax(1)
    acc = float((pred == y_leaf).mean())
    return dict(leaf_acc=acc, abstain_rate=abstain, pred=pred)


# ============================================================
# FedAvg-tree: cabeza en árbol + pérdida marginalizada, pero SIN máscara
# (agregación estándar de todos los clientes en cada nodo). Aísla si la
# ganancia de ResolvFL viene de la máscara o solo de usar la taxonomía.
# ============================================================
def run_fedavg_tree(tax, clients, cfg: FedConfig, device, ckpt_path=None,
                    progress_every=5, backbone_factory=None):
    if backbone_factory is None:
        backbone_factory = ConvBackbone
    backbone = backbone_factory(cfg.feat_dim).to(device)
    head = TreeHead(tax, cfg.feat_dim).to(device)
    n_samples = [len(c["y_node"]) for c in clients]

    start_round = 0
    if ckpt_path is not None:
        start_round, _ = load_checkpoint(ckpt_path, backbone, head, map_location=device)
        if start_round > 0:
            print(f"    [checkpoint] retomando FedAvg-tree desde la ronda {start_round}")

    t0 = time.time()
    for rnd in range(start_round, cfg.rounds):
        bb_list, hd_list, losses = [], [], []
        for c in clients:
            bb, hd = copy.deepcopy(backbone), copy.deepcopy(head)
            l = local_train_tree(tax, bb, hd, c, cfg, device)
            losses.append(l)
            bb_list.append({k: v.detach().clone() for k, v in bb.state_dict().items()})
            hd_list.append({k: v.detach().clone() for k, v in hd.state_dict().items()})
        backbone.load_state_dict(plain_average(bb_list, n_samples))
        head.load_state_dict(plain_average(hd_list, n_samples))  # UNMASKED: every node averaged over all clients

        if ckpt_path is not None and (rnd + 1) % cfg.save_every == 0:
            save_checkpoint(ckpt_path, rnd + 1, backbone, head)
        if rnd % progress_every == 0 or rnd == cfg.rounds - 1:
            elapsed = time.time() - t0
            print(f"    [FedAvg-tree] ronda {rnd+1:3d}/{cfg.rounds} | "
                  f"loss medio={np.mean(losses):.4f} | {elapsed:.0f}s transcurridos")

    if ckpt_path is not None:
        save_checkpoint(ckpt_path, cfg.rounds, backbone, head)
    return backbone, head


In [ ]:
import numpy as np
import torch
from resolvfl_colab import (load_aves_features, FeatureBackbone, TreeHead, FlatHead,
                            double_dirichlet_partition, FedConfig,
                            run_resolvfl, run_flat_fedavg, eval_tree, eval_flat)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tax, X, y_leaf = load_aves_features(FEATURES_PATH)
print(f"Taxonomía real de Aves: {len(tax.leaves)} especies (hojas), "
      f"{len(tax.internal)-1} nodos internos (+ROOT), profundidad {tax.max_depth}")
assert len(tax.leaves) == 1486, f"esperaba 1486 especies, salieron {len(tax.leaves)}"
print(f"X: {X.shape}  (vectores de {X.shape[1]} dim, ya extraídos con ResNet-50 congelado)")

# Reparto real de tamaño por nivel: cuántas especies tiene cada orden
order_sizes = {}
for u in tax.internal:
    name = tax.id2name[u]
    depth = tax.depth[u]
    if depth == 1:  # nivel orden
        order_sizes[name] = len(tax.leaves_under[u])
print(f"\n{len(order_sizes)} órdenes de aves. Reparto de especies por orden (desbalanceo real):")
for name, n in sorted(order_sizes.items(), key=lambda kv: -kv[1])[:8]:
    print(f"  {name:20s}: {n:4d} especies")
print(f"  ... (y {max(0,len(order_sizes)-8)} órdenes más)")


## Federated partition and initial training

Construct the class/granularity double-Dirichlet partition for the 1,486-species Aves taxonomy.

In [ ]:
K = 10
rng = np.random.default_rng(1)
clients = double_dirichlet_partition(tax, X, y_leaf, K=K, alpha_cls=0.5, alpha_dep=0.5, rng=rng)
for i, c in enumerate(clients):
    print(f"  cliente {i}: {len(c['y_node'])} muestras")

# Split train/test con SEMILLA PROPIA, independiente de la de la particion.
# (Antes compartian el mismo objeto rng -- el split dependia de cuantos
# numeros aleatorios habia consumido la particion antes, lo que lo hacia
# fragil a cualquier cambio de codigo o de orden de ejecucion. Con una
# semilla dedicada, Xte/yte quedan fijos siempre, pase lo que pase antes.)
rng_split = np.random.default_rng(999)
n = len(X)
idx = rng_split.permutation(n)
split = int(0.9 * n)
train_idx, test_idx = idx[:split], idx[split:]
Xte, yte = X[test_idx], y_leaf[test_idx]
print(f"\nTest: {len(Xte)} muestras (split con semilla independiente, fija)")


In [ ]:
cfg = FedConfig(rounds=150, local_epochs=1, batch_size=64, lr=0.01,
                feat_dim=128, seed=1, save_every=10)

ckpt_fg = os.path.join(CKPT_DIR, "resolvfl_aves.pt")
backbone_factory = lambda d: FeatureBackbone(in_dim=X.shape[1], feat_dim=d)
backbone_fg, head_fg = run_resolvfl(tax, clients, cfg, device, ckpt_path=ckpt_fg,
                                    backbone_factory=backbone_factory)
res_fg = eval_tree(tax, backbone_fg, head_fg, Xte, yte, device)
print(f"\n>>> ResolvFL — accuracy en especies (test): {res_fg['leaf_acc']:.4f}")


In [ ]:
ckpt_fl = os.path.join(CKPT_DIR, "flat_fedavg_aves.pt")
backbone_fl, head_fl = run_flat_fedavg(tax, clients, cfg, device, ckpt_path=ckpt_fl,
                                    backbone_factory=backbone_factory)
res_fl = eval_flat(tax, backbone_fl, head_fl, Xte, yte, device)
print(f"\n>>> FedAvg-flat — accuracy en especies (test): {res_fl['leaf_acc']:.4f}")
print(f">>> FedAvg-flat — tasa de abstención a nodo interno: {res_fl['abstain_rate']:.4f}")


## Tree baseline

Use the same architecture and path loss as ResolvFL, aggregating every head over all clients. Compare Flat, Tree and ResolvFL on the shared partition.

In [ ]:
from resolvfl_colab import run_fedavg_tree

ckpt_tree = os.path.join(CKPT_DIR, "fedavg_tree_aves.pt")
backbone_tree, head_tree = run_fedavg_tree(tax, clients, cfg, device,
                                           ckpt_path=ckpt_tree,
                                           backbone_factory=backbone_factory)
res_tree = eval_tree(tax, backbone_tree, head_tree, Xte, yte, device)
print(f"\n>>> FedAvg-tree (sin máscara) — accuracy en especies (test): {res_tree['leaf_acc']:.4f}")

# Añadir al resumen (merge, no sobrescribir)
_sp = os.path.join(CKPT_DIR, "results_summary.json")
_s = {}
if os.path.exists(_sp):
    with open(_sp) as f:
        _s = json.load(f)
_s["fedavg_tree_leaf_acc"] = res_tree["leaf_acc"]
with open(_sp, "w") as f:
    json.dump(_s, f, indent=2)
print("Guardado en Drive.")


In [ ]:
# Cargamos lo que ya hubiera en Drive de sesiones anteriores y AÑADIMOS a
# eso, en vez de reemplazarlo -- así el resultado de E1 (si ya se corrió)
# no se pierde al re-ejecutar esta celda en una sesión nueva.
_summary_path = os.path.join(CKPT_DIR, "results_summary.json")
summary = {}
if os.path.exists(_summary_path):
    with open(_summary_path) as f:
        summary = json.load(f)

summary.update({
    "dataset": "iNat2021-mini-Aves",
    "n_species": len(tax.leaves),
    "n_internal_nodes": len(tax.internal) - 1,
    "max_depth": tax.max_depth,
    "resolvfl_leaf_acc": res_fg["leaf_acc"],
    "flat_fedavg_leaf_acc": res_fl["leaf_acc"],
    "flat_fedavg_abstain_rate": res_fl["abstain_rate"],
    "rounds": cfg.rounds,
    "num_clients": K,
})
with open(_summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print("Resumen guardado en Drive:")
for k, v in summary.items():
    print(f"  {k}: {v}")


## Empirical coverage audit and intervention

Count observed path supervision at internal nodes, then remove descendant supervision in the target subtree. Finite coverage counts describe this partition; they are not a population identifiability guarantee.

In [ ]:
hists = []
for c in clients:
    vals, cnts = np.unique(c["y_node"], return_counts=True)
    hists.append({int(v): cnt / len(c["y_node"]) for v, cnt in zip(vals, cnts)})
cert = tax.certificate(hists)
n_unresolved = sum(1 for v in cert["resolved"].values() if not v)
print(f"Nodos sin resolver con la partición actual: {n_unresolved} / {len(cert['resolved'])}")
if n_unresolved > 0:
    print("Nombres:", [tax.id2name[u] for u in cert["maximal_unresolved"]][:10])


In [ ]:
# Elegir un nodo real defendible para violar: un genero (profundidad 3)
# con al menos 3 especies, ni el mas grande ni el mas pequeno -- para que
# no parezca elegido a dedo, cogemos el que quede mas cerca de la MEDIANA
# de tamaño entre los generos con >=3 especies.
genus_candidates = []
for u in tax.internal:
    if tax.depth[u] == 3:  # nivel genero (ROOT=0, orden=1, familia=2, genero=3)
        n_leaves = len(tax.leaves_under[u])
        if n_leaves >= 3:
            genus_candidates.append((tax.id2name[u], n_leaves, u))

genus_candidates.sort(key=lambda t: t[1])
median_idx = len(genus_candidates) // 2
VIOLATED_NAME, VIOLATED_SIZE, VIOLATED_ID = genus_candidates[median_idx]
print(f"Genero elegido (mediana de tamaño entre {len(genus_candidates)} candidatos "
      f"con >=3 especies): '{VIOLATED_NAME}', {VIOLATED_SIZE} especies")
print("Especies bajo ese genero:",
     [tax.id2name[l] for l in tax.leaves_under[VIOLATED_ID]])


In [ ]:
def censor_below_node(node_id, clients_list, tax):
    under_node = set(tax.leaves_under[node_id])
    out = []
    for c in clients_list:
        y = c["y_node"].copy()
        for i, yn in enumerate(y):
            if yn in under_node or yn == node_id:
                y[i] = node_id
        out.append(dict(c, y_node=y))
    return out, under_node

clients_violated, under_node = censor_below_node(VIOLATED_ID, clients, tax)

hists_v = []
for c in clients_violated:
    vals, cnts = np.unique(c["y_node"], return_counts=True)
    hists_v.append({int(v): cnt / len(c["y_node"]) for v, cnt in zip(vals, cnts)})
cert_v = tax.certificate(hists_v)
unresolved_names = [tax.id2name[u] for u in cert_v["maximal_unresolved"]]
print("Nodos sin resolver detectados por el certificado:", unresolved_names)
assert unresolved_names == [VIOLATED_NAME], f"esperaba solo {VIOLATED_NAME}, salio {unresolved_names}"
print("OK: el certificado detecta exactamente el nodo censurado, antes de entrenar nada.")


## Restart intervention training for trajectory logging

This optional cell deletes the intervention checkpoint and starts that run again to record intermediate evaluations. Review the checkpoint path before execution.

In [ ]:
import os
# Solo borra y fuerza reentrenamiento si la trayectoria NO esta ya completa
# en Drive. Si ya la capturaste en una sesion anterior (10 puntos), no
# reentrena: se salta este paso y reutiliza el checkpoint existente.
ckpt_v = os.path.join(CKPT_DIR, f"resolvfl_violated_{VIOLATED_NAME}.pt")
traj_path = os.path.join(CKPT_DIR, f"e1_trajectory_{VIOLATED_NAME}.json")

_traj_ok = False
if os.path.exists(traj_path):
    import json as _json
    with open(traj_path) as _f:
        _t = _json.load(_f)
    # trayectoria completa = tiene un punto en la ultima ronda (cfg.rounds)
    if _t and max(p["round"] for p in _t) >= cfg.rounds:
        _traj_ok = True

if _traj_ok:
    print("Trayectoria ya completa en Drive (", len(_t), "puntos) -- NO se reentrena.")
else:
    for p in [ckpt_v, traj_path]:
        if os.path.exists(p):
            os.remove(p)
            print("Borrado (para capturar trayectoria):", p)
        else:
            print("No existia:", p)


In [ ]:
# Entrenamiento del modelo violado CON trayectoria: evalua las 3 metricas
# de E1 cada 15 rondas, no solo al final -- coste extra minimo (un forward
# pass sobre el test set), guardado incremental por si Colab corta.
from resolvfl_colab import TreeHead, local_train_tree, masked_aggregate, \
    local_supervision_mass, load_checkpoint, save_checkpoint
import copy, time

violated_idx = [tax.leaf_index[l] for l in under_node]
other_idx = [tax.leaf_index[l] for l in tax.leaves if l not in under_node]

def block_acc(pred, y, idxs):
    mask = np.isin(y, idxs)
    return float((pred[mask] == y[mask]).mean()) if mask.sum() > 0 else float("nan")

def superclass_acc_from_pred(pred, y_true, idxs):
    mask = np.isin(y_true, idxs)
    if mask.sum() == 0:
        return float("nan")
    return float(np.isin(pred[mask], idxs).mean())

TRAJ_EVAL_EVERY = 15
ckpt_v = os.path.join(CKPT_DIR, f"resolvfl_violated_{VIOLATED_NAME}.pt")
traj_path = os.path.join(CKPT_DIR, f"e1_trajectory_{VIOLATED_NAME}.json")

trajectory = []
if os.path.exists(traj_path):
    with open(traj_path) as f:
        trajectory = json.load(f)
    print(f"Trayectoria previa recuperada: {len(trajectory)} puntos")

torch.manual_seed(cfg.seed)  # reproducibilidad si hay que retomar tras un corte
backbone_v = backbone_factory(cfg.feat_dim).to(device)
head_v = TreeHead(tax, cfg.feat_dim).to(device)
masses_v = [local_supervision_mass(tax, c["y_node"]) for c in clients_violated]
n_samples_v = [len(c["y_node"]) for c in clients_violated]

start_round, _ = load_checkpoint(ckpt_v, backbone_v, head_v, map_location=device)
if start_round > 0:
    print(f"[checkpoint] retomando desde la ronda {start_round}")

t0 = time.time()
for rnd in range(start_round, cfg.rounds):
    states = []
    for c in clients_violated:
        bb, hd = copy.deepcopy(backbone_v), copy.deepcopy(head_v)
        local_train_tree(tax, bb, hd, c, cfg, device)
        states.append(({k: v.detach().clone() for k, v in bb.state_dict().items()},
                       {k: v.detach().clone() for k, v in hd.state_dict().items()}))
    bb_sd, hd_sd = masked_aggregate(tax, states, n_samples_v, masses_v)
    backbone_v.load_state_dict(bb_sd)
    head_v.load_state_dict(hd_sd)

    if (rnd + 1) % cfg.save_every == 0:
        save_checkpoint(ckpt_v, rnd + 1, backbone_v, head_v)

    if (rnd + 1) % TRAJ_EVAL_EVERY == 0 or rnd == cfg.rounds - 1:
        res_v_t = eval_tree(tax, backbone_v, head_v, Xte, yte, device)
        point = {
            "round": rnd + 1,
            "leaf_acc_overall": res_v_t["leaf_acc"],
            "violated_leaves_acc": block_acc(res_v_t["pred"], yte, violated_idx),
            "other_leaves_acc": block_acc(res_v_t["pred"], yte, other_idx),
            "superclass_acc": superclass_acc_from_pred(res_v_t["pred"], yte, violated_idx),
        }
        elapsed = time.time() - t0
        print(f"[ResolvFL violado] ronda {rnd+1:3d}/{cfg.rounds} | "
             f"violado={point['violated_leaves_acc']:.4f} | "
             f"otras={point['other_leaves_acc']:.4f} | "
             f"superclase={point['superclass_acc']:.4f} | {elapsed:.0f}s")
        trajectory.append(point)
        with open(traj_path, "w") as f:
            json.dump(trajectory, f, indent=2)

save_checkpoint(ckpt_v, cfg.rounds, backbone_v, head_v)
backbone_v_final, head_v_final = backbone_v, head_v
res_v = eval_tree(tax, backbone_v_final, head_v_final, Xte, yte, device)  # para las celdas siguientes
print(f"\nTrayectoria completa guardada en Drive: {traj_path}")
print(f"Puntos evaluados en rondas: {[p['round'] for p in trajectory]}")


In [ ]:
violated_idx = [tax.leaf_index[l] for l in under_node]
other_idx = [tax.leaf_index[l] for l in tax.leaves if l not in under_node]

def block_acc(pred, y, idxs):
    mask = np.isin(y, idxs)
    return float((pred[mask] == y[mask]).mean()) if mask.sum() > 0 else float("nan")

acc_viol_ref = block_acc(res_fg["pred"], yte, violated_idx)
acc_viol_v = block_acc(res_v["pred"], yte, violated_idx)
acc_other_ref = block_acc(res_fg["pred"], yte, other_idx)
acc_other_v = block_acc(res_v["pred"], yte, other_idx)
mask_v = np.isin(yte, violated_idx)
superclass_acc_ref = float(np.isin(res_fg["pred"][mask_v], violated_idx).mean()) if mask_v.sum() else float("nan")
superclass_acc_v = float(np.isin(res_v["pred"][mask_v], violated_idx).mean()) if mask_v.sum() else float("nan")

print(f"=============== E1 en iNat-Aves: genero '{VIOLATED_NAME}' ===============")
print(f"Especies de {VIOLATED_NAME} (violadas):")
print(f"   run resoluble: {acc_viol_ref:.4f}  |  run violado: {acc_viol_v:.4f}")
print(f"Las otras {len(other_idx)} especies (resueltas):")
print(f"   run resoluble: {acc_other_ref:.4f}  |  run violado: {acc_other_v:.4f}")
print(f"Identificacion a nivel de familia para muestras de {VIOLATED_NAME}:")
print(f"   run resoluble: {superclass_acc_ref:.4f}  |  run violado: {superclass_acc_v:.4f}")

summary["E1_aves"] = dict(
    violated_node=VIOLATED_NAME, violated_size=VIOLATED_SIZE,
    violated_leaves_resolvable=acc_viol_ref, violated_leaves_violated=acc_viol_v,
    other_leaves_resolvable=acc_other_ref, other_leaves_violated=acc_other_v,
    family_acc_resolvable=superclass_acc_ref, family_acc_violated=superclass_acc_v,
)
summary["E1_aves_trajectory"] = trajectory
with open(os.path.join(CKPT_DIR, "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("\nGuardado en Drive (incluye trayectoria de", len(trajectory), "puntos).")


## Additional main-comparison seeds

Run Flat, Tree and ResolvFL for seed labels 2 and 3, each with a separately sampled partition. Seed 1 is stored in the earlier summary. Each run uses 150 rounds.

In [ ]:
import torch

MAIN_SEEDS = [2, 3]  # la semilla 1 ya está hecha (resultados actuales en summary)
multiseed_path = os.path.join(CKPT_DIR, "multiseed_main.json")

multiseed_results = {}
if os.path.exists(multiseed_path):
    with open(multiseed_path) as f:
        multiseed_results = json.load(f)
    print(f"Progreso previo recuperado: semillas {list(multiseed_results.keys())}")

def _save_multiseed():
    with open(multiseed_path, "w") as f:
        json.dump(multiseed_results, f, indent=2)

for seed in MAIN_SEEDS:
    seed_key = str(seed)
    if seed_key in multiseed_results and all(
        k in multiseed_results[seed_key] for k in ["flat", "tree", "resolvfl"]
    ):
        print(f"[seed {seed}] ya completa, se salta.")
        continue

    print(f"\n=== Semilla {seed} ===")
    rng_s = np.random.default_rng(seed)
    clients_s = double_dirichlet_partition(tax, X, y_leaf, K=K, alpha_cls=0.5,
                                           alpha_dep=0.5, rng=rng_s)
    cfg_s = FedConfig(rounds=150, local_epochs=1, batch_size=64, lr=0.01,
                      feat_dim=128, seed=seed, save_every=10)

    seed_result = multiseed_results.get(seed_key, {})
    multiseed_results[seed_key] = seed_result  # referencia compartida, para poder guardar tras cada modelo

    if "flat" not in seed_result:
        torch.manual_seed(seed)
        ck = os.path.join(CKPT_DIR, f"flat_fedavg_aves_seed{seed}.pt")
        bb, hd = run_flat_fedavg(tax, clients_s, cfg_s, device, ckpt_path=ck,
                                 backbone_factory=backbone_factory)
        r = eval_flat(tax, bb, hd, Xte, yte, device)
        seed_result["flat"] = {"leaf_acc": r["leaf_acc"], "abstain_rate": r["abstain_rate"]}
        print(f"  flat: acc={r['leaf_acc']:.4f} abstain={r['abstain_rate']:.4f}")
        _save_multiseed()

    if "tree" not in seed_result:
        torch.manual_seed(seed)
        ck = os.path.join(CKPT_DIR, f"fedavg_tree_aves_seed{seed}.pt")
        bb, hd = run_fedavg_tree(tax, clients_s, cfg_s, device, ckpt_path=ck,
                                 backbone_factory=backbone_factory)
        r = eval_tree(tax, bb, hd, Xte, yte, device)
        seed_result["tree"] = {"leaf_acc": r["leaf_acc"]}
        print(f"  tree: acc={r['leaf_acc']:.4f}")
        _save_multiseed()

    if "resolvfl" not in seed_result:
        torch.manual_seed(seed)
        ck = os.path.join(CKPT_DIR, f"resolvfl_aves_seed{seed}.pt")
        bb, hd = run_resolvfl(tax, clients_s, cfg_s, device, ckpt_path=ck,
                              backbone_factory=backbone_factory)
        r = eval_tree(tax, bb, hd, Xte, yte, device)
        seed_result["resolvfl"] = {"leaf_acc": r["leaf_acc"]}
        print(f"  resolvfl: acc={r['leaf_acc']:.4f}")
        _save_multiseed()

    print(f"[seed {seed}] guardada en Drive.")

print("\nCompleto:", list(multiseed_results.keys()))


In [ ]:
# Resumen AUTONOMO: reconstruye sus propias rutas y carga 'summary' desde
# su propio JSON, no depende de que ninguna otra celda se haya ejecutado
# antes. Se puede correr esta celda sola, en cualquier momento.
import os, json
import numpy as np

_CKPT_DIR = "/content/drive/MyDrive/resolvfl_inat_aves"
_multiseed_path = os.path.join(_CKPT_DIR, "multiseed_main.json")
_summary_path = os.path.join(_CKPT_DIR, "results_summary.json")

if not os.path.exists(_summary_path):
    print("Todavía no hay resumen guardado -- nada que mostrar aún.")
else:
    with open(_summary_path) as f:
        _summary = json.load(f)
    _r = {}
    if os.path.exists(_multiseed_path):
        with open(_multiseed_path) as f:
            _r = json.load(f)

    _seed1 = {"flat": {"leaf_acc": _summary.get("flat_fedavg_leaf_acc")},
             "tree": {"leaf_acc": _summary.get("fedavg_tree_leaf_acc")},
             "resolvfl": {"leaf_acc": _summary.get("resolvfl_leaf_acc")}}
    _all = {"1": _seed1, **_r}

    print("=== Tabla principal iNat, por semilla ===\n")
    for seed in sorted(_all, key=int):
        r = _all[seed]
        print(f"Semilla {seed}:")
        if "flat" in r and r["flat"].get("leaf_acc") is not None:
            ab = r["flat"].get("abstain_rate")
            ab_str = f"  abstain={ab:.4f}" if ab is not None else ""
            print(f"  flat:     acc={r['flat']['leaf_acc']:.4f}{ab_str}")
        if "tree" in r and r["tree"].get("leaf_acc") is not None:
            print(f"  tree:     acc={r['tree']['leaf_acc']:.4f}")
        if "resolvfl" in r and r["resolvfl"].get("leaf_acc") is not None:
            print(f"  resolvfl: acc={r['resolvfl']['leaf_acc']:.4f}")
        print()

    complete = {s: r for s, r in _all.items()
               if all(k in r and r[k].get("leaf_acc") is not None for k in ["flat","tree","resolvfl"])}
    if len(complete) >= 2:
        print(f"=== Media +/- std muestral, ddof=1 ({len(complete)} semillas completas: {sorted(complete, key=int)}) ===\n")
        for method in ["flat", "tree", "resolvfl"]:
            vals = [complete[s][method]["leaf_acc"] for s in complete]
            print(f"  {method}: {np.mean(vals)*100:.2f} +/- {np.std(vals, ddof=1)*100:.2f}")
    else:
        print(f"Todavía solo {len(complete)} semilla(s) completa(s) -- faltan más para media/std.")


## Paired Petroica intervention

Within each seed, reuse the partition and initialization for reference and intervention. Replace descendant annotations in the five-species Petroica subtree with coarser labels. Historical ancestor-of-decoded-leaf diagnostics remain distinct from marginal-prefix accuracy.

In [ ]:
PAIRED_SEEDS = [10, 20, 30]  # semillas nuevas, distintas de las de la tabla principal
paired_path = os.path.join(CKPT_DIR, "paired_violation_petroica.json")

paired_results = []
if os.path.exists(paired_path):
    with open(paired_path) as f:
        paired_results = json.load(f)
    print(f"Progreso previo recuperado: {len(paired_results)} semillas")

done_seeds = {p["seed"] for p in paired_results}

for seed in PAIRED_SEEDS:
    if seed in done_seeds:
        print(f"[seed {seed}] ya completa, se salta.")
        continue

    print(f"\n=== Semilla emparejada {seed} ===")
    rng_s = np.random.default_rng(seed)
    clients_s = double_dirichlet_partition(tax, X, y_leaf, K=K, alpha_cls=0.5,
                                           alpha_dep=0.5, rng=rng_s)
    clients_v_s, under_node_s = censor_below_node(VIOLATED_ID, clients_s, tax)
    violated_idx_s = [tax.leaf_index[l] for l in under_node_s]
    other_idx_s = [tax.leaf_index[l] for l in tax.leaves if l not in under_node_s]

    cfg_s = FedConfig(rounds=150, local_epochs=1, batch_size=64, lr=0.01,
                      feat_dim=128, seed=seed, save_every=10)

    # Resoluble: misma particion, seed fijada justo antes de entrenar
    torch.manual_seed(seed)
    ck_r = os.path.join(CKPT_DIR, f"paired_resolvable_seed{seed}.pt")
    bb_r, hd_r = run_resolvfl(tax, clients_s, cfg_s, device, ckpt_path=ck_r,
                              backbone_factory=backbone_factory)
    res_r = eval_tree(tax, bb_r, hd_r, Xte, yte, device)

    # Violado: MISMA seed (misma inicializacion), solo cambia clients_v_s
    torch.manual_seed(seed)
    ck_v = os.path.join(CKPT_DIR, f"paired_violated_seed{seed}.pt")
    bb_v, hd_v = run_resolvfl(tax, clients_v_s, cfg_s, device, ckpt_path=ck_v,
                              backbone_factory=backbone_factory)
    res_v = eval_tree(tax, bb_v, hd_v, Xte, yte, device)

    point = {
        "seed": seed,
        "violated_resolvable": block_acc(res_r["pred"], yte, violated_idx_s),
        "violated_violated": block_acc(res_v["pred"], yte, violated_idx_s),
        "other_resolvable": block_acc(res_r["pred"], yte, other_idx_s),
        "other_violated": block_acc(res_v["pred"], yte, other_idx_s),
        "family_resolvable": superclass_acc_from_pred(res_r["pred"], yte, violated_idx_s),
        "family_violated": superclass_acc_from_pred(res_v["pred"], yte, violated_idx_s),
    }
    paired_results.append(point)
    with open(paired_path, "w") as f:
        json.dump(paired_results, f, indent=2)
    vr, vv = point["violated_resolvable"], point["violated_violated"]
    orr, ov = point["other_resolvable"], point["other_violated"]
    print(f"  violado: {vr:.4f} -> {vv:.4f}")
    print(f"  otras:   {orr:.4f} -> {ov:.4f}")
    print(f"[seed {seed}] guardada en Drive.")

print("\nCompleto:", [p["seed"] for p in paired_results])


In [ ]:
# Resumen AUTONOMO del protocolo emparejado -- reconstruye su propia ruta,
# no depende de ninguna celda anterior. 'genero' (no 'familia': la métrica
# es identificación a nivel de género, historical decoded-leaf-ancestor diagnostic; not marginal-prefix accuracy).
import os, json
import numpy as np

_CKPT_DIR = "/content/drive/MyDrive/resolvfl_inat_aves"
_paired_path = os.path.join(_CKPT_DIR, "paired_violation_petroica.json")

if not os.path.exists(_paired_path):
    print("Todavía no hay resultados del protocolo emparejado -- nada que mostrar aún.")
else:
    with open(_paired_path) as f:
        _pr = json.load(f)

    print(f"=== Protocolo emparejado Petroica, {len(_pr)} semilla(s) ===\n")
    for p in _pr:
        print(f"Semilla {p['seed']}:")
        print(f"  violado: {p['violated_resolvable']:.4f} -> {p['violated_violated']:.4f}  "
             f"(diff {p['violated_violated']-p['violated_resolvable']:+.4f})")
        print(f"  otras:   {p['other_resolvable']:.4f} -> {p['other_violated']:.4f}  "
             f"(diff {p['other_violated']-p['other_resolvable']:+.4f})")
        print(f"  genero:  {p['family_resolvable']:.4f} -> {p['family_violated']:.4f}  "
             f"(diff {p['family_violated']-p['family_resolvable']:+.4f})")
        print()

    if len(_pr) >= 2:
        print(f"=== Diferencias emparejadas, media +/- std muestral ddof=1 ({len(_pr)} semillas) ===\n")
        for key in ["violated", "other", "family"]:
            diffs = [p[f"{key}_violated"] - p[f"{key}_resolvable"] for p in _pr]
            etiqueta = "genero" if key == "family" else key
            print(f"  {etiqueta}: {np.mean(diffs)*100:+.2f} +/- {np.std(diffs, ddof=1)*100:.2f} puntos")


## Active-cohort supervision weight

Compute rho(u), the sum of client weights over the active cohort at each internal node, from the saved partition seeds. This descriptive diagnostic does not train models and is distinct from Jaccard overlap.

In [ ]:
def compute_rho(tax, clients):
    n_samples = [len(c["y_node"]) for c in clients]
    total = float(sum(n_samples))
    w = [nk / total for nk in n_samples]
    masses = [local_supervision_mass(tax, c["y_node"]) for c in clients]
    rho = {}
    for u in tax.internal:
        cohort_w = [w[k] for k in range(len(clients)) if masses[k].get(u, 0.0) > 0]
        rho[u] = sum(cohort_w)
    return rho

# Beneficio de masking ya observado por semilla (resolvfl - tree), calculado antes
masking_benefit = {1: 0.0677, 2: 0.0015, 3: -0.0032}

print("=== rho(u): solapamiento de cohortes por semilla ===\n")
rho_summary = {}
for seed in [1, 2, 3]:
    rng_s = np.random.default_rng(seed)
    clients_s = double_dirichlet_partition(tax, X, y_leaf, K=K, alpha_cls=0.5,
                                           alpha_dep=0.5, rng=rng_s)
    rho = compute_rho(tax, clients_s)
    vals = list(rho.values())
    rho_summary[seed] = {"mean": float(np.mean(vals)), "median": float(np.median(vals)),
                         "min": float(min(vals)), "max": float(max(vals))}
    print(f"semilla {seed} (beneficio masking={masking_benefit[seed]*100:+.2f}pts): "
         f"rho(u) media={np.mean(vals):.3f} mediana={np.median(vals):.3f} "
         f"min={min(vals):.3f} max={max(vals):.3f}  (n={len(vals)} nodos internos)")

with open(os.path.join(CKPT_DIR, "rho_cohort_overlap.json"), "w") as f:
    json.dump(rho_summary, f, indent=2)
print("\nGuardado en Drive: rho_cohort_overlap.json")
